<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Slices and fairness

**[Slices and fairness](https://learning.nextia-ai.com/courses/anomaly-detection/m07/slices-and-fairness/)** · Anomaly Detection: Find What Is Unusual · Module 7, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** check a detector's false alerts group by group (account type, country, device, region and customer age), see when a group is too small to trust, and explain why a blocked honest customer matters more than one average shows.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | The alert budget of 50 a day, from [The alert budget](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/); the 95% interval of a share, from [Evaluate without a full answer key](https://learning.nextia-ai.com/courses/anomaly-detection/m07/evaluate-without-a-full-answer-key/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m07/slices-and-fairness/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M07-L04-slices-and-fairness/slices-and-fairness-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A tiny example: ten customers

Ten customers each made 100 honest payments. Eight are personal customers, two are business customers. The table counts the false alerts of each one.

> **Predict.** Are the two business customers close to the average of all ten?

> **Check.** You should see 260 per 10,000 for all customers, 62.5 for personal customers and 1,050 for business customers.

In [ ]:
tiny = pd.DataFrame({
    "customer": ["personal"] * 8 + ["business"] * 2,
    "honest payments": [100] * 10,
    "false alerts": [1, 0, 1, 1, 0, 1, 0, 1, 12, 9],
})
total = tiny[["honest payments", "false alerts"]].sum()
print(f"all customers: {total['false alerts'] / total['honest payments'] * 10_000:.0f} false alerts per 10,000 honest payments")
by_group = tiny.groupby("customer")[["honest payments", "false alerts"]].sum()
by_group["per 10,000"] = by_group["false alerts"] / by_group["honest payments"] * 10_000
by_group

One average of 260 hides a group that gets about 17 times more false alerts than the other (1,050 against 62.5).

## 2. Isolation Forest, 50 alerts a day

The same detector as in the last two lessons. An **honest payment** is one that is not planted. A **false alert** is an alert on an honest payment.

> **Check.** You should see 166,986 honest payments and 136.2 false alerts per 10,000.

In [ ]:
import compare, features
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(scaler.transform(behaviour.loc[reference.index]))
score = -forest.score_samples(scaler.transform(behaviour.loc[evaluation.index]))
alerts = compare.daily_alerts(evaluation, score, 50)
alert = alerts.set_index("payment_id").reviewed.reindex(evaluation.payment_id).to_numpy()
planted = evaluation.payment_id.isin(compare.planted_payments().payment_id).to_numpy()
honest = ~planted
print(f"{honest.sum():,} honest payments, {(alert & honest).sum():,} of them got an alert: "
      f"{(alert & honest).sum() / honest.sum() * 10_000:.1f} per 10,000")

## 3. A slice: business and personal customers

A **slice** is one group of the data, here the payments of one account type. `false_alerts_by()` gives the rate of each slice, with its counts.

> **Check.** You should see 444.1 for business customers and 47.5 for personal customers.

In [ ]:
def false_alerts_by(groups, flags=None):
    """False alerts per 10,000 honest payments, for each group; with the counts, so small groups show.
    flags: one True/False per evaluation payment (an alert or not); by default, Isolation Forest's top 50 a day."""
    flags = alert if flags is None else flags
    t = pd.DataFrame({"group": np.asarray(groups)[honest], "alert": flags[honest]})
    out = t.groupby("group").alert.agg(false_alerts="sum", honest_payments="size")
    out["per 10,000"] = (out.false_alerts / out.honest_payments * 10_000).round(1)
    return out.sort_values("per 10,000", ascending=False)

false_alerts_by(evaluation.account_type)

## 4. A small slice: payments made abroad

> **Check.** You should see 661.8 per 10,000 abroad, from only 272 honest payments, with a wide interval: 366 to 957.

In [ ]:
paid_abroad = np.where(evaluation.country == "CA", "in Canada", "abroad")
t = false_alerts_by(paid_abroad)
share = t.false_alerts / t.honest_payments
margin = 1.96 * np.sqrt(share * (1 - share) / t.honest_payments)
t["95% interval per 10,000"] = [f"{(s - m) * 10_000:.0f} to {(s + m) * 10_000:.0f}" for s, m in zip(share, margin)]
t

## 5. More slices: device, region and customer age

Not every slice you expect to be high is high. Look at the new customers (their first 30 days) and at the new region, `east`.

In [ ]:
first_payment = payments.groupby("customer_id").ts.min()
age_days = (evaluation.ts - evaluation.customer_id.map(first_payment)).dt.days
for name, groups in [("device", evaluation.device_type), ("region", evaluation.region),
                     ("customer age", np.where(age_days < 30, "first 30 days", "older"))]:
    print(name); print(false_alerts_by(groups).to_string(), end="\n\n")

Does the detector also find more fraud among business customers? Run the next cell.

In [ ]:
biz = (evaluation.account_type == "business").to_numpy()
for name, m in [("business", biz), ("personal", ~biz)]:
    print(f"{name}: {(planted & m & alert).sum()} of {(planted & m).sum()} planted payments found")

Count the customers behind the false alerts. Are they the same customers again and again?

In [ ]:
false = evaluation[alert & honest]
per_customer = false.groupby("customer_id").size()
account = evaluation.groupby("customer_id").account_type.first()
pd.DataFrame({"customers": account.value_counts(),
              "with a false alert": account[per_customer.index].value_counts(),
              "with 3 or more": account[per_customer[per_customer >= 3].index].value_counts(),
              "most for one customer": per_customer.groupby(account[per_customer.index]).max()})

## 6. Check yourself

**Your turn 1.** Slice the false alerts by `merchant_category`. Write the category with the highest rate as `worst_category`.

In [ ]:
worst_category = ...   # your code

In [ ]:
expect_hash('worst_category', worst_category, '52b79833efcff956')

**Your turn 2.** Night payments are those from 00:00 to 05:59 (`hour` from 0 to 5). Write the false alerts per 10,000 honest night payments as `night_per_10000`.

In [ ]:
night_per_10000 = ...   # your code

In [ ]:
expect('night_per_10000', night_per_10000, 841.5, tolerance=0.001)

## 7. Module practice: defend a detector

Compare Isolation Forest and kNN distance at the team's 50 alerts a day: precision at the budget, cost per week, and the business slice. Then write three sentences for Sofia: which detector, its precision and cost, and one slice result with what you will do about it.

In [ ]:
from sklearn.neighbors import NearestNeighbors
X_ref = scaler.transform(behaviour.loc[reference.index])
sample = X_ref[np.random.default_rng(0).choice(len(X_ref), 20_000, replace=False)]   # kNN, as in Module 4
dist, _ = NearestNeighbors(n_neighbors=10).fit(sample).kneighbors(scaler.transform(behaviour.loc[evaluation.index]))
knn = dist[:, -1]
table = compare.Table()
table.add(compare.score_payments(evaluation, score, "Isolation Forest", "behaviour features, 50 a day"))
table.add(compare.score_payments(evaluation, knn, "kNN distance", "behaviour features, 50 a day"))
table.show()
knn_alert = compare.daily_alerts(evaluation, knn, 50).set_index("payment_id").reviewed.reindex(evaluation.payment_id).to_numpy()
false_alerts_by(evaluation.account_type, knn_alert)

## Recap

- One average of false alerts can hide a group that gets many times more.
- Report each slice with its counts, and an interval when it is small.
- A false alert on an honest customer can block a payment. Count those customers apart from the money.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [From unsupervised to semi-supervised](https://learning.nextia-ai.com/courses/anomaly-detection/m08/from-unsupervised-to-semi-supervised/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m07/slices-and-fairness/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.